# 미션 4-2 · 한국어 도메인 감성어 30개 선정하기 (3단계)

> **목표:** 2단계에서 저장한 **형태소 빈도·KNU 매칭 CSV**를 `pandas`로 탐색하고, **영화 리뷰에 특화된 감성 표현 30개 이상**을 근거를 기록하면서 선정한다.

## 이번 노트북의 작업 범위

- **입력 A:** `data/outputs/sentiment_vocab/nsmc_train_okt_stem_knu_*.csv` — 2단계 분석 결과
- **입력 B:** `data/datasets/nsmc/ratings_train.txt` — 리뷰 원문 + 긍정/부정 정답(`label`)
- **입력 C:** `data/lexicons/ko/SentiWord_info.json` — KNU 원본 사전(존재·구조 확인 및 비교용)
- **출력:** `data/outputs/sentiment_vocab/`의 **레이블별 출현 통계**, **후보 검토 CSV**
- **최종 초안(선택):** `data/lexicons/ko/custom_sentiment_draft.json`
- **하지 않는 일:** 감성 점수 자동 부여, KNU 원본 덮어쓰기, `ratings_test.txt` 사용

## 왜 상위 30개로는 부족했을까?

- `영화`, `연기`, `스토리` → **영화 도메인의 주제어**이지만 단독으로는 대체로 감성 방향이 없다.
- `너무`, `정말` → 감성 단어가 아니라 **강조·정도 표현**일 수 있다.
- `않다` → 감성어보다 **부정 처리 규칙**에서 다루는 것이 적절하다.
- `있다`, `같다` → 빈도가 높지만 **문맥에 따라 역할과 의미가 바뀌는 범용어**다.
- `unmatched` → **2단계 단일 형태소 매칭에서 발견되지 않음**이라는 뜻이며, KNU에 의미상 완전히 없다는 증거는 아니다.

**중심 원칙:** 빈도와 정답 레이블의 연관성은 *검토 순서*를 정하는 근거다. 실제 감성 점수는 **사용 문맥을 보고 사람이 결정**한다.

## 0. 실행 준비 및 프로젝트 경로

- 노트북 위치: `4-2/notebooks/`
- 모든 원본 및 결과 데이터: `4-2/data/`
- Python 커널: `4-2/.venv/bin/python`
- 필요한 Python 패키지: `pandas`, `konlpy`, `JPype1` (Okt 실행 시 Java 필요)
- **`pandas` 미설치 시** 프로젝트 루트 터미널에서 `.venv/bin/python -m pip install pandas` 실행
- 노트북은 **설치·다운로드를 수행하지 않는다**. 데이터가 없으면 해당 파일을 생성하는 이전 단계로 돌아간다.

In [ ]:
from pathlib import Path
import json
import os
import sys
from collections import Counter, defaultdict
from IPython.display import display

try:
    import pandas as pd
except ImportError as e:
    raise ImportError("pandas가 필요합니다: 프로젝트 루트에서 .venv/bin/python -m pip install pandas") from e

# VS Code/Jupyter의 실행 디렉터리가 루트일 수도 notebooks/일 수도 있다.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("4-2/ 또는 4-2/notebooks/에서 실행해야 합니다.")
ROOT = ROOT.resolve()
DATA = ROOT / "data"
OUT = DATA / "outputs" / "sentiment_vocab"
OUT.mkdir(parents=True, exist_ok=True)
KNU_PATH = DATA / "lexicons" / "ko" / "SentiWord_info.json"
TRAIN_PATH = DATA / "datasets" / "nsmc" / "ratings_train.txt"

# Java 설정은 Okt import 이전에 수행한다 (현재 노트북 커널에만 적용).
if not os.environ.get("JAVA_HOME"):
    for java_home in (DATA / "jdk-25" / "Contents" / "Home", DATA / "jdk-25"):
        if (java_home / "bin" / "java").is_file():
            os.environ["JAVA_HOME"] = str(java_home)
            os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
            break

print("프로젝트 루트:", ROOT)
print("실행 Python:", sys.executable)
print("pandas 버전:", pd.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "시스템 설정 사용"))
print("출력 위치:", OUT.relative_to(ROOT))
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    print("⚠ 루트 .venv가 아닌 커널입니다. 패키지 오류가 나면 커널 경로를 확인하세요.")

for required in (KNU_PATH, TRAIN_PATH):
    if not required.is_file():
        raise FileNotFoundError(f"필수 입력이 없습니다: {required}")

## 1. 2단계 집계 CSV와 KNU 감성 사전 로드

- `full` 파일을 우선 사용한다. 표본 파일만 있다면 아래의 `SOURCE_SCOPE`를 명시적으로 바꾼다.
- `knu_score`의 **결측값(`NaN`)과 0점은 다른 값**이다.
- 집계 단위는 `(token, pos)`이며, 서로 다른 품사에 속하는 같은 문자열은 별도 행이다.
- KNU JSON은 원본이 유지되고 있는지만 확인한다. **2단계에서 만든 매칭 결과를 그대로 사용**하므로 여기서 매칭을 새로 계산하지 않는다.

In [ ]:
# 2단계에서 MAX_REVIEWS=None이었다면 "full" CSV가 만들어졌을 것이다.
SOURCE_SCOPE = "full"              # 또는 "first_3000" 등의 이전 단계 출력 접미사
VOCAB_CSV = OUT / f"nsmc_train_okt_stem_knu_{SOURCE_SCOPE}.csv"
if not VOCAB_CSV.is_file():
    available = sorted(p.name for p in OUT.glob("nsmc_train_okt_stem_knu_*.csv"))
    raise FileNotFoundError(
        f"{VOCAB_CSV.name}이 없습니다. 2단계 노트북에서 먼저 생성하세요.\n사용 가능한 파일: {available}"
    )

with KNU_PATH.open("r", encoding="utf-8-sig") as f:
    knu_raw = json.load(f)

vocab = pd.read_csv(VOCAB_CSV, encoding="utf-8-sig")
required_cols = {"token", "pos", "token_count", "review_count", "knu_score", "match_status"}
missing = required_cols.difference(vocab.columns)
if missing:
    raise ValueError(f"집계 CSV에 필요한 컬럼이 없습니다: {sorted(missing)}")

vocab["token"] = vocab["token"].astype("string")
vocab["pos"] = vocab["pos"].astype("string")
vocab["match_status"] = vocab["match_status"].astype("string")
for col in ("token_count", "review_count"):
    vocab[col] = pd.to_numeric(vocab[col], errors="raise").astype("int64")
vocab["knu_score"] = pd.to_numeric(vocab["knu_score"], errors="coerce")  # NaN은 미확정, 0은 유효 점수
vocab["repeat_per_review"] = vocab["token_count"] / vocab["review_count"].replace(0, float("nan"))

print(f"KNU 원본 항목: {len(knu_raw):,}개")
print(f"집계된 (형태소, 품사): {len(vocab):,}개")
print("2단계 결과 파일:", VOCAB_CSV.relative_to(ROOT))
display(vocab.head(10))

## 2. pandas로 전체 분포 살펴보기

- `match_status`별 고유 단어 수와 출현 리뷰 수를 집계한다.
- 품사(`pos`)별로 어디에 미등록 표현이 많이 있는지 확인한다.
- **`unmatched`만 집중 탐색하되** `ambiguous`(점수 충돌)도 별도 확인할 가치가 있다.
- `review_count`가 높은 단어가 반드시 감성어인 것은 아니다.

In [ ]:
by_status = (
    vocab.groupby("match_status", dropna=False)
    .agg(words=("token", "size"), token_occurrences=("token_count", "sum"), review_hits=("review_count", "sum"))
    .sort_values("words", ascending=False)
)
print("[매칭 상태별 요약]")
display(by_status)

by_pos = (
    vocab.groupby(["pos", "match_status"], dropna=False)
    .agg(words=("token", "size"), review_hits=("review_count", "sum"))
    .reset_index()
    .sort_values(["review_hits", "words"], ascending=False)
)
print("[품사 × 매칭 상태 (상위 30행)]")
display(by_pos.head(30))

## 3. Top 30을 넘어 **Top N** 범위 탐색

- `TOP_N`을 `100`, `300`, `1000` 등으로 바꾸면서 탐색한다.
- `token_count`: **같은 리뷰 내 중복도 포함**한 출현 횟수.
- `review_count`: **같은 리뷰에서 여러 번 등장해도 한 번** 센 리뷰 수.
- `POS_FILTER`: 품사별로 나누면 주제 명사보다 형용사·동사 쪽 감성 표현을 찾기 쉽다.
- `MIN_CHARS`와 `MIN_REVIEWS`는 **탐색 범위 조절용**이다. 감성 여부의 판정 규칙이 아니다.

In [ ]:
# ===== 이 셀에서 탐색 조건을 자유롭게 변경 =====
TOP_N = 100
POS_FILTER = ["Adjective", "Verb", "Noun", "Adverb", "Exclamation"]  # None이면 품사 제한 없음
MIN_CHARS = 2
MIN_REVIEWS = 10
STATUS_FILTER = "unmatched"  # None=전체, "unmatched"=미등록, "ambiguous"=점수 충돌

DISPLAY_COLS = ["token", "pos", "token_count", "review_count",
                "repeat_per_review", "knu_score", "match_status"]


def filter_vocab(frame, pos_filter=POS_FILTER, min_chars=MIN_CHARS,
                 min_reviews=MIN_REVIEWS, status=STATUS_FILTER):
    """빈도표에서 사람이 확인하고 싶은 부분만 추린다. 원본 DataFrame은 변경하지 않는다."""
    mask = frame["token"].notna() & (frame["token"].str.len() >= min_chars)
    mask &= frame["review_count"] >= min_reviews
    if pos_filter is not None:
        mask &= frame["pos"].isin(pos_filter)
    if status is not None:
        mask &= frame["match_status"].eq(status)
    return frame.loc[mask].copy()


def top_table(frame, by="review_count", n=TOP_N):
    if by not in {"token_count", "review_count"}:
        raise ValueError("by는 token_count 또는 review_count 중 하나여야 합니다.")
    return (frame.sort_values([by, "token"], ascending=[False, True])
                 .head(n).reset_index(drop=True))

filtered = filter_vocab(vocab)
print(f"조건에 맞는 단어: {len(filtered):,}개 / 전체 {len(vocab):,}개")
print(f"[리뷰 수 기준 Top {TOP_N}]")
display(top_table(filtered, "review_count")[DISPLAY_COLS])
print(f"[전체 등장 횟수 기준 Top {TOP_N}]")
display(top_table(filtered, "token_count")[DISPLAY_COLS])

## 4. 특정 품사·빈도 구간·문자열을 직접 조회

`pandas`에서 반복해서 사용할 조작은 다음과 같다.

- `.loc[조건, 열목록]` : 필요한 행과 열만 선택
- `.isin([...])` : 품사 여러 개를 묶어서 선택
- `.between(a, b)` : 빈도의 상·하한 범위 지정
- `.str.contains(...)` : 단어에 포함된 부분 문자열 검색 (`regex=False`는 일반 문자열 검색)
- `.sort_values(...)` : 원하는 지표로 순위 재정렬

아래 조건을 바꿔가며 **빈도가 너무 높은 범용어를 넘어서** 탐색한다.

In [ ]:
# 예시 1. '형용사'이면서 KNU 미등록, 리뷰 수 20~2000인 단어
adjectives = vocab.loc[
    vocab["pos"].eq("Adjective")
    & vocab["match_status"].eq("unmatched")
    & vocab["review_count"].between(20, 2000)
]
display(adjectives.sort_values("review_count", ascending=False)[DISPLAY_COLS].head(60))

# 예시 2. 관심 어근 검색: 이 문자열을 자유롭게 바꿔보자.
SEARCH_TEXT = "재미"
found = vocab.loc[vocab["token"].str.contains(SEARCH_TEXT, regex=False, na=False)]
display(found.sort_values("review_count", ascending=False)[DISPLAY_COLS +
    (["sample_review"] if "sample_review" in vocab.columns else [])].head(30))

## 5. 빈도 외의 단서: 긍정·부정 리뷰에서 얼마나 다르게 등장하는가?

상위 빈도표만 보면 `영화`, `연기`, `정말` 같은 단어가 앞을 차지한다. 그러므로 **레이블별 출현 빈도**를 추가로 계산한다.

- NSMC `label=1`: 긍정 리뷰 / `label=0`: 부정 리뷰
- `positive_review_hits`: 긍정 리뷰 중 해당 형태소가 등장한 리뷰 수
- `negative_review_hits`: 부정 리뷰 중 해당 형태소가 등장한 리뷰 수
- **리뷰당 1회만 집계**하여 반복 표현이 통계를 지배하지 않도록 한다.
- 분석기가 2단계와 동일한 `Okt.pos(norm=True, stem=True)`를 사용하므로 **새로 형태소 분석하는 작업**이 필요하다.
- 부하를 줄이기 위해 처음에는 **클래스별 균형 표본**을 사용한다. 나중에 `LABEL_SAMPLE_SIZE=None`으로 전체 train을 분석할 수 있다.
- `ratings_test.txt`는 **후보 선택에 사용하지 않는다**.

**주의:** 레이블 연관성이 높아도 그 단어 자체의 감성이라는 뜻은 아니다. 예를 들어 특정 배우 이름이나 영화 제목도 리뷰 레이블과 상관관계를 가질 수 있다.

In [ ]:
train = pd.read_csv(TRAIN_PATH, sep="\t", encoding="utf-8", dtype={"id": "string", "document": "string"})
assert {"id", "document", "label"}.issubset(train.columns), train.columns
train = train.dropna(subset=["document", "label"]).copy()
train["document"] = train["document"].str.strip()
train = train.loc[train["document"].str.len() > 0].copy()
train["label"] = pd.to_numeric(train["label"], errors="raise").astype("int64")
assert set(train["label"].unique()).issubset({0, 1}), "NSMC label은 0/1이어야 합니다."

print("분석 대상: NSMC train (원본 test는 미사용)")
print("전체 유효 리뷰:", len(train))
display(train["label"].value_counts().rename(index={0: "부정(0)", 1: "긍정(1)"}).to_frame("reviews"))
display(train.head(5))

In [ ]:
# ===== 레이블별 통계에 사용할 분석 범위 =====
LABEL_SAMPLE_SIZE = 20_000  # 빠른 확인: 2000 / 전체 분석: None
RANDOM_STATE = 42
MAX_CONTEXT_EXAMPLES_PER_LABEL = 3
CONTEXT_POOL_SIZE = 5000  # 리뷰 예시를 수집할 미등록 단어 개수(빈도순)
PROGRESS_EVERY = 2000

# 클래스별로 비슷한 수를 무작위로 추출한다.
# 표본으로 계산된 비율은 '표본 내 추정치'이며, 전체 데이터를 분석한 값과 다를 수 있다.
if LABEL_SAMPLE_SIZE is None:
    work = train.copy()
    scope_label = "full"
else:
    if LABEL_SAMPLE_SIZE < 2:
        raise ValueError("LABEL_SAMPLE_SIZE는 2 이상이거나 None이어야 합니다.")
    n_neg = LABEL_SAMPLE_SIZE // 2
    n_pos = LABEL_SAMPLE_SIZE - n_neg
    neg = train.loc[train["label"].eq(0)].sample(
        n=min(n_neg, (train["label"] == 0).sum()), random_state=RANDOM_STATE
    )
    pos = train.loc[train["label"].eq(1)].sample(
        n=min(n_pos, (train["label"] == 1).sum()), random_state=RANDOM_STATE
    )
    work = pd.concat([neg, pos], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE)
    scope_label = f"balanced_{LABEL_SAMPLE_SIZE}_seed{RANDOM_STATE}"

# 캐시 파일: 같은 설정이면 Okt 형태소 분석을 반복하지 않는다.
LABEL_CSV = OUT / f"label_evidence_okt_stem_{scope_label}_v1.csv"
EXAMPLES_JSON = OUT / f"label_context_examples_{scope_label}_v1.json"
META_JSON = OUT / f"label_evidence_meta_{scope_label}_v1.json"

# 실제 문맥을 모을 대상은 '2단계 미등록 표현' 중 빈도 상위 일부다.
context_candidates = (
    filter_vocab(vocab, pos_filter=["Noun", "Verb", "Adjective", "Adverb", "Exclamation"],
                 min_chars=2, min_reviews=2, status="unmatched")
    .sort_values("review_count", ascending=False).head(CONTEXT_POOL_SIZE)
)
CONTEXT_KEYS = set(zip(context_candidates["token"], context_candidates["pos"]))

print("레이블별 통계 분석 예정 리뷰:", len(work))
print("문맥 예시 수집 대상 형태소·품사:", len(CONTEXT_KEYS))
print("캐시 경로:", LABEL_CSV.relative_to(ROOT))
print("기존 통계 캐시:", "있음" if LABEL_CSV.is_file() else "없음")

## 6. 형태소별 긍정·부정 출현 통계 생성 및 캐싱

- 아래 셀은 **통계 캐시가 없다면** KoNLPy Okt를 호출한다.
- `token_count`와 `review_count`를 계산한 2단계와 동일하게 `norm=True, stem=True`로 분석한다.
- `set(tokens)`을 사용해 **같은 리뷰 내 중복은 1회로 집계**한다.
- 긍정·부정 문맥 예시를 최대 3개씩 별도로 남긴다(상위 후보에 대해서만).
- 결과를 CSV/JSON으로 저장하므로 **재실행 시 형태소 분석을 생략**한다.
- 분석 범위나 품사 정책을 바꾼 경우에는 **캐시 파일명(`v1`)을 변경하거나 기존 캐시를 삭제**해야 한다.

In [ ]:
if LABEL_CSV.is_file() and META_JSON.is_file() and EXAMPLES_JSON.is_file():
    print("기존 레이블별 통계 캐시를 재사용합니다.")
    label_evidence = pd.read_csv(LABEL_CSV, encoding="utf-8-sig")
    meta = json.loads(META_JSON.read_text(encoding="utf-8"))
    context_examples = json.loads(EXAMPLES_JSON.read_text(encoding="utf-8"))
else:
    from konlpy.tag import Okt
    okt = Okt()
    per_label = {0: Counter(), 1: Counter()}
    context_examples = {}  # key: '형태소\t품사' -> {'0': [...], '1': [...]}
    analyzed_by_label = Counter()
    failures = 0

    for i, row in enumerate(work.itertuples(index=False), start=1):
        try:
            parsed = [
                (form, pos)
                for form, pos in okt.pos(row.document, norm=True, stem=True)
                if form.strip() and pos not in {"Punctuation", "Space"}
            ]
        except Exception as e:
            failures += 1
            if failures <= 3:
                print("형태소 분석 실패 예시:", type(e).__name__, str(e)[:100])
            continue

        label = int(row.label)
        analyzed_by_label[label] += 1
        uniq = set(parsed)
        per_label[label].update(uniq)

        # 표본에서 일부 문맥만 저장. 해당 단어가 포함된 리뷰를 의미한다.
        for token, pos in uniq & CONTEXT_KEYS:
            key = f"{token}\t{pos}"
            entries = context_examples.setdefault(key, {"0": [], "1": []})[str(label)]
            if len(entries) < MAX_CONTEXT_EXAMPLES_PER_LABEL:
                entries.append({"id": str(row.id), "text": str(row.document)[:500]})

        if PROGRESS_EVERY and i % PROGRESS_EVERY == 0:
            print(f"{i:,} / {len(work):,}개 분석 시도")

    rows = []
    for token, pos in per_label[0].keys() | per_label[1].keys():
        rows.append({
            "token": token, "pos": pos,
            "positive_review_hits": per_label[1][(token, pos)],
            "negative_review_hits": per_label[0][(token, pos)],
        })
    label_evidence = pd.DataFrame(
        rows, columns=["token", "pos", "positive_review_hits", "negative_review_hits"]
    )
    label_evidence.to_csv(LABEL_CSV, index=False, encoding="utf-8-sig")
    meta = {
        "sample_scope": scope_label, "source": "NSMC ratings_train.txt",
        "okt_params": {"norm": True, "stem": True},
        "positive_analyzed": analyzed_by_label[1],
        "negative_analyzed": analyzed_by_label[0],
        "failed_reviews": failures,
        "context_pool_size": CONTEXT_POOL_SIZE,
    }
    META_JSON.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    EXAMPLES_JSON.write_text(json.dumps(context_examples, ensure_ascii=False, indent=2), encoding="utf-8")

# 캐시를 읽은 경우에도 반드시 저장된 표본 크기를 사용한다.
for c in ("positive_review_hits", "negative_review_hits"):
    label_evidence[c] = pd.to_numeric(label_evidence[c], errors="raise").astype("int64")
print("분석한 긍정 리뷰:", f"{meta['positive_analyzed']:,}")
print("분석한 부정 리뷰:", f"{meta['negative_analyzed']:,}")
print("분석 실패:", meta.get("failed_reviews", 0))
print("레이블 통계 행 수:", len(label_evidence))
display(label_evidence.head(10))

## 7. pandas 병합 후 긍정·부정 **연관성 지표** 계산

기존 CSV와 `(token, pos)`로 병합하여 표본 내 통계를 추가한다.

- `positive_review_hits`, `negative_review_hits`: 각각 긍정·부정 리뷰 중 **등장한 리뷰 개수**
- `positive_prevalence` = 긍정 리뷰 중 해당 단어가 나온 비율
- `negative_prevalence` = 부정 리뷰 중 해당 단어가 나온 비율
- `log2_class_ratio` = 두 비율의 비에 로그2를 취한 값 (0.5회 평활화 적용)
  - **양수**: 긍정 리뷰에 상대적으로 자주 등장
  - **음수**: 부정 리뷰에 상대적으로 자주 등장
  - **0 근처**: 양쪽에 비슷하게 등장
- **주의:** 로그 비율은 **감성 점수가 아니고**, 통계적 연관성을 보여주는 지표다.
- 두 클래스의 전체 리뷰 개수가 다르더라도 빈도 비율로 비교한다.

In [ ]:
import numpy as np

n_pos = int(meta["positive_analyzed"])
n_neg = int(meta["negative_analyzed"])
if n_pos <= 0 or n_neg <= 0:
    raise ValueError("긍정/부정 리뷰가 모두 분석되지 않았습니다. 표본 및 형태소 분석 설정을 확인하세요.")

# 빈도 CSV는 full일 수 있지만 레이블 통계는 별도의 표본일 수 있다.
# 아래 결합은 두 출처의 집계 범위를 절대로 섞어 같은 분모로 계산하지 않는다.
merged = vocab.merge(label_evidence, on=["token", "pos"], how="left", validate="one_to_one")
for c in ("positive_review_hits", "negative_review_hits"):
    merged[c] = merged[c].fillna(0).astype("int64")
merged["label_sample_review_hits"] = merged["positive_review_hits"] + merged["negative_review_hits"]
merged["positive_prevalence"] = merged["positive_review_hits"] / n_pos
merged["negative_prevalence"] = merged["negative_review_hits"] / n_neg

# 평활화는 샘플에 등장하지 않은 클래스도 비교할 수 있게 해준다.
smoothed_pos = (merged["positive_review_hits"] + 0.5) / (n_pos + 1)
smoothed_neg = (merged["negative_review_hits"] + 0.5) / (n_neg + 1)
merged["log2_class_ratio"] = np.log2(smoothed_pos / smoothed_neg)
merged["direction_in_sample"] = np.select(
    [merged["log2_class_ratio"] >= 0.8, merged["log2_class_ratio"] <= -0.8],
    ["긍정 리뷰 편중", "부정 리뷰 편중"],
    default="약한 편중/유사",
)

EVIDENCE_CSV = OUT / f"nsmc_token_knu_with_label_evidence_{SOURCE_SCOPE}_{scope_label}.csv"
merged.to_csv(EVIDENCE_CSV, index=False, encoding="utf-8-sig")
print("결합 데이터 저장:", EVIDENCE_CSV.relative_to(ROOT))
print("전체 토큰 수:", len(merged), "/ 레이블 표본에서 등장한 토큰:", (merged["label_sample_review_hits"] > 0).sum())
display(merged[["token", "pos", "review_count", "knu_score", "match_status",
                "positive_review_hits", "negative_review_hits", "log2_class_ratio"]].head(10))

## 8. 긍정 쪽 / 부정 쪽으로 편중된 **후보를 따로 조회**

- **빈도 상위 표**만 보면 주제어가 대부분이었으므로, 이제 클래스별 **상대 출현 빈도 차이**로 정렬한다.
- `MIN_SAMPLE_HITS`: 표본에서 너무 드물게 나온 표현을 제외하여 극단적인 비율을 완화한다.
- `MIN_ABS_LOG2_RATIO`: 편중이 일정 수준 이상인 단어만 확인한다.
- 감성 단어로의 **추가 여부와 점수는 여전히 사람이 문맥으로 판정**한다.
- 필요하면 `N=100`으로 넓게 보거나, `pos_filter=["Adjective"]`로 좁혀보자.

In [ ]:
MIN_SAMPLE_HITS = 15
MIN_ABS_LOG2_RATIO = 0.8
N = 60

EVIDENCE_COLS = ["token", "pos", "review_count", "positive_review_hits",
                 "negative_review_hits", "positive_prevalence",
                 "negative_prevalence", "log2_class_ratio", "knu_score", "match_status"]


def evidence_candidates(frame, pos_filter=None, min_hits=MIN_SAMPLE_HITS,
                        min_abs_ratio=MIN_ABS_LOG2_RATIO, status="unmatched"):
    mask = (frame["label_sample_review_hits"] >= min_hits)
    mask &= frame["log2_class_ratio"].abs() >= min_abs_ratio
    if status is not None:
        mask &= frame["match_status"].eq(status)
    if pos_filter is not None:
        mask &= frame["pos"].isin(pos_filter)
    mask &= frame["token"].str.len().ge(2)
    return frame.loc[mask].copy()

focused = evidence_candidates(merged, pos_filter=["Adjective", "Verb", "Noun", "Adverb", "Exclamation"])
print("[긍정 리뷰 편중 Top N] — 감성 점수 자동 할당 아님")
display(focused.loc[focused["log2_class_ratio"] > 0].sort_values("log2_class_ratio", ascending=False)[EVIDENCE_COLS].head(N))
print("[부정 리뷰 편중 Top N] — 감성 점수 자동 할당 아님")
display(focused.loc[focused["log2_class_ratio"] < 0].sort_values("log2_class_ratio", ascending=True)[EVIDENCE_COLS].head(N))

## 9. 실제 문맥에서 단어가 어떻게 쓰였는지 보기

빈도·레이블 연관성만으로 결정할 수 없는 상황을 검토하는 단계다.

- 같은 단어의 **긍정 리뷰 문맥 1~3개**와 **부정 리뷰 문맥 1~3개**를 나란히 본다.
- `"연기"`처럼 높은 빈도라도 양쪽에서 다르게 쓰이면 기본 점수를 주지 않는 편이 안전하다.
- `"나쁘지 않다"`처럼 부정 규칙의 영향을 받는 표현은 **단어 자체 점수와 문장 규칙을 혼동하지 않는다**.
- 예문은 앞서 지정한 `CONTEXT_POOL_SIZE` 상위 토큰의 표본에서 수집되므로, 없는 예문은 **미등장 또는 예시 수집 대상 밖**이라는 뜻이다.
- 리뷰 레이블은 **리뷰 전체에 대한 평가**이고, 특정 단어의 감성에 대한 정답은 아니다.

In [ ]:
def inspect_word(token, pos=None):
    """후보 1개를 조회하고 표본에 수집된 긍정·부정 리뷰 예시를 보여준다."""
    hits = merged.loc[merged["token"].eq(token)]
    if pos is not None:
        hits = hits.loc[hits["pos"].eq(pos)]
    if hits.empty:
        print(f"'{token}' 검색 결과 없음 (2단계 CSV 범위 밖일 수 있음)")
        return
    display(hits[EVIDENCE_COLS + (["sample_review"] if "sample_review" in merged else [])])
    for row in hits.itertuples(index=False):
        key = f"{row.token}\t{row.pos}"
        found = context_examples.get(key, {"0": [], "1": []})
        print(f"\n[{row.token} / {row.pos}] 긍정 리뷰 예문")
        for ex in found.get("1", []):
            print(" +", ex["text"])
        print(f"[{row.token} / {row.pos}] 부정 리뷰 예문")
        for ex in found.get("0", []):
            print(" -", ex["text"])
        if not found.get("0") and not found.get("1"):
            print("   (수집된 예문 없음: 표본 또는 문맥 수집 대상 범위를 늘려보세요)")

# 먼저 표에서 관심 있는 단어로 바꿔 실행해 보자.
inspect_word("연기", pos="Noun")

## 10. 30개 선정을 위한 검토 표를 CSV로 만들기

**의도적으로 감성 점수는 채우지 않는다.** 검토할 근거만 한 행에 모은다.

- 후보 풀 = **미등록 단어의 상위 빈도 후보** + **긍정 편중 후보** + **부정 편중 후보**
- `(token, pos)`로 중복 제거하고, 점수가 이미 매칭된 KNU 단어는 기본적으로 제외한다.
- 수작업으로 작성할 열:
  - `decision`: `add`(사전 추가), `reject`(제외), `review`(보류)
  - `role`: `sentiment`(감성 표현), `aspect`(주제/대상), `modifier`(강조), `negator`(부정), `other`
  - `custom_score`: 직접 결정한 KNU 호환 점수 `-2`, `-1`, `0`, `1`, `2`
  - `reason`: 감성 의미 및 실제 리뷰 근거
- `영화`, `스토리`, `연기`처럼 **주제어만으로 점수를 주지 않는다**.
- 파일이 이미 존재하면 **덮어쓰지 않는다.** 사람이 채운 기록을 보존하기 위해서다.

In [ ]:
CURATION_CSV = OUT / f"domain_lexicon_curation_{SOURCE_SCOPE}_{scope_label}.csv"

# 빈도/편중을 기준으로 서로 다른 유형의 후보를 폭넓게 확보한다.
base_pool = filter_vocab(
    merged, pos_filter=["Noun", "Adjective", "Verb", "Adverb", "Exclamation"],
    min_chars=2, min_reviews=5, status="unmatched"
)
frequency_pool = base_pool.nlargest(400, "review_count")
positive_pool = focused.loc[focused["log2_class_ratio"] > 0].nlargest(250, "log2_class_ratio")
negative_pool = focused.loc[focused["log2_class_ratio"] < 0].nsmallest(250, "log2_class_ratio")

candidate_pool = (pd.concat([frequency_pool, positive_pool, negative_pool], ignore_index=True)
                  .drop_duplicates(subset=["token", "pos"])
                  .sort_values(["review_count", "token"], ascending=[False, True])
                  .reset_index(drop=True))

if not CURATION_CSV.exists():
    review = candidate_pool.copy()
    review["decision"] = "review"
    review["role"] = ""
    review["custom_score"] = ""
    review["reason"] = ""
    cols = ["token", "pos", "review_count", "token_count", "match_status", "knu_score",
            "positive_review_hits", "negative_review_hits", "log2_class_ratio",
            "decision", "role", "custom_score", "reason"]
    review[cols].to_csv(CURATION_CSV, index=False, encoding="utf-8-sig")
    print("신규 검토 파일 생성:", CURATION_CSV.relative_to(ROOT))
else:
    print("기존 검토 파일 유지 (덮어쓰지 않음):", CURATION_CSV.relative_to(ROOT))

curation = pd.read_csv(CURATION_CSV, encoding="utf-8-sig")
print("검토 후보 수:", len(curation))
display(curation.head(15))

## 11. 검토 CSV를 pandas로 수정하고 다시 저장하기

- `CURATION_CSV` 파일을 편집기로 직접 편집해도 되고, 아래처럼 pandas `.loc`을 사용해도 된다.
- **예제 수정 코드는 주석 처리**되어 있다. 검토 없이 임의 단어에 감성 점수를 매기지 않는다.
- `(token, pos)`를 함께 지정해 동일 문자열·다른 품사의 충돌을 방지한다.
- 저장 전에 변경된 행을 다시 출력해서 확인한다.

```python
# 실제 문맥 확인 후에만 주석을 풀고 값 수정하기
mask = curation["token"].eq("검토한_단어") & curation["pos"].eq("Adjective")
curation.loc[mask, ["decision", "role", "custom_score", "reason"]] = [
    "add", "sentiment", -2, "부정 문맥에서 반복적으로 사용됨(예문 확인)"
]
curation.to_csv(CURATION_CSV, index=False, encoding="utf-8-sig")
```

In [ ]:
# 먼저 몇 개가 선택되었는지 확인한다. 아무것도 선택되지 않았어도 정상이다.
curation["decision"] = curation["decision"].fillna("review").astype("string")
selected = curation.loc[curation["decision"].eq("add")].copy()
print(f"현재 추가 대상으로 표시한 항목: {len(selected)}개 / 목표 30개 이상")
if not selected.empty:
    display(selected[["token", "pos", "custom_score", "role", "reason"]].head(50))

# 필요하면 이 셀 위의 예제를 복사해 수정하고 아래를 실행한다.
# curation.to_csv(CURATION_CSV, index=False, encoding="utf-8-sig")

## 12. 선정 기록 검증 및 사용자 정의 사전 초안 내보내기

- `decision == "add"`와 `role == "sentiment"`인 행만 사전 후보로 인정한다.
- `custom_score`가 누락되었거나 `-2 ~ 2`의 정수가 아니면 **저장하지 않는다**.
- **0점은 중립 표현**이라는 뜻이므로, 이번에 목표로 하는 **감성 표현 30개**의 충족 개수에는 포함하지 않는다.
- 2단계에서 `unmatched`라고 나왔더라도, 형태소의 다른 표기나 KNU의 복합 표현으로 이미 존재할 수 있으니 최종 병합 전에 점검해야 한다.
- 30개에 못 미쳐도 지금까지의 검토 CSV는 그대로 남는다. **30개를 억지로 채우지 않는다.**
- 초안 JSON은 **KNU 원본과 별개**이며 감성 분석기에 자동으로 병합하지 않는다.

In [ ]:
# 최종 후보로 인정하기 위한 보수적인 검증
curation = pd.read_csv(CURATION_CSV, encoding="utf-8-sig")
curation["decision"] = curation["decision"].fillna("review").astype("string")
curation["role"] = curation["role"].fillna("").astype("string")
curation["reason"] = curation["reason"].fillna("").astype("string")
curation["custom_score"] = pd.to_numeric(curation["custom_score"], errors="coerce")

to_add = curation.loc[
    curation["decision"].eq("add") & curation["role"].eq("sentiment")
].copy()
errors = []
if to_add[["token", "pos"]].duplicated().any():
    errors.append("동일한 (token, pos)가 중복 선택되었습니다.")
if to_add["custom_score"].isna().any():
    errors.append("감성 점수가 비어 있는 선택 항목이 있습니다.")
if not to_add.empty and not to_add["custom_score"].dropna().isin([-2, -1, 0, 1, 2]).all():
    errors.append("감성 점수는 -2, -1, 0, 1, 2 중 하나여야 합니다.")
if to_add["reason"].str.strip().eq("").any():
    errors.append("선정 근거(reason)가 없는 항목이 있습니다.")

nonzero = to_add.loc[to_add["custom_score"].isin([-2, -1, 1, 2])]
print(f"선택한 감성어: {len(to_add)}개 / 그중 0점 아닌 감성어: {len(nonzero)}개")
if errors:
    print("검증 오류:")
    for error in errors:
        print(" -", error)
elif len(nonzero) < 30:
    print("30개 미만이므로 JSON 초안을 내보내지 않습니다. 검토 CSV를 이어서 편집하세요.")
else:
    export_path = DATA / "lexicons" / "ko" / "custom_sentiment_draft.json"
    entries = [
        {"word": str(row.token), "pos": str(row.pos),
         "polarity": int(row.custom_score), "reason": str(row.reason)}
        for row in nonzero.itertuples(index=False)
    ]
    if export_path.exists():
        print("기존 수작업 초안을 보호하기 위해 자동 덮어쓰지 않습니다:", export_path.relative_to(ROOT))
    else:
        export_path.write_text(json.dumps(entries, ensure_ascii=False, indent=2), encoding="utf-8")
        print("초안 생성:", export_path.relative_to(ROOT))
    print("주의: 이 JSON은 검토용 초안이며 KNU 사전에 병합된 상태가 아닙니다.")

## 13. 학습 결과 정리

### 이 노트북에서 만든 것

- `nsmc_token_knu_with_label_evidence_*.csv`: 2단계 집계 + 표본의 긍정·부정 리뷰 출현 통계
- `label_evidence_okt_stem_*.csv`: 표본에서 형태소별 클래스 출현 집계(재분석 회피용 캐시)
- `label_context_examples_*.json`: 긍정·부정 리뷰 문맥 예시(검토용 캐시)
- `domain_lexicon_curation_*.csv`: **사람이 직접 점수와 근거를 기록하는 후보 표**
- `custom_sentiment_draft.json`: 유효한 비영(非0) 감성 표현 **30개 이상**을 검토한 뒤 생성하는 선택 결과 초안

### 실제 선정 시 구분할 것

- **감성 표현:** 문맥에 따라 긍정·부정 평가를 실질적으로 나타내는 단어/표현
- **영화 도메인 주제어:** 영화, 배우, 연기 등. 대개 점수 없이 보존
- **강조어:** 너무, 정말 등. 나중에 감성점수 가중치 규칙으로 처리
- **부정어:** 안, 않다 등. 나중에 부정 범위·반전 규칙으로 처리
- **문맥 의존 표현:** '미쳤다'처럼 상황에 따라 긍정·부정이 바뀌는 표현은 무조건 점수 부여하지 않음

### 다음 실험에서 비교할 것

- KNU 사전만 사용할 때의 정확도/F1
- KNU + 도메인 감성어를 추가했을 때의 정확도/F1
- 그 다음에 부정어·강조어 규칙을 추가했을 때의 정확도/F1

**원칙:** 최종 평가 데이터(`ratings_test.txt`)는 이 후보 선정 과정에 사용하지 않는다.